In [1]:
from strands import Agent, tool
from typing import Dict, Any, List
import boto3
import numpy as np
import re
import faiss
import os
import time
import json
from strands.models import BedrockModel
from dotenv import load_dotenv
from scripts.agents.prompts import (
    TEXT_TO_SQL_PROMPT_V4,
    AGENT_ORCHESTRATOR_SYSTEM_PROMPT_V4,
    FALLBACK_PROMPT_V1,
    RAG_PROMPT_V1,
)

In [3]:
load_dotenv()

AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")
REGION_NAME = os.environ.get("AWS_REGION")
S3_BUCKET = "talk-to-your-data-bucket"
S3_KEY = "queries_rag_sql.json"

In [4]:
# Boto3 session
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="us-east-1",
)

# Bedrock model custom session
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    boto_session=session,
    temperature=0.3,
    top_p=0.8,
)

## RAG to SQL

In [5]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("RAG-Notebook")

In [9]:
s3 = session.client("s3", region_name=REGION_NAME)
bedrock_rt = session.client("bedrock-runtime", region_name=REGION_NAME)

In [ ]:
# Embedding Titan
TITAN_EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBED_DIM = 1024


def titan_embed(text: str, dims: int = EMBED_DIM, normalize: bool = True) -> np.ndarray:
    """
    Gera embedding com Titan v2 (Bedrock Runtime).
    """
    body = json.dumps({"inputText": text, "dimensions": dims, "normalize": normalize})
    resp = bedrock_rt.invoke_model(
        modelId=TITAN_EMBED_MODEL_ID,
        body=body,
        accept="application/json",
        contentType="application/json",
    )
    out = json.loads(resp["body"].read())
    emb = np.array(out["embedding"], dtype=np.float32)
    return emb


# Smoke test (opcional):
e = titan_embed("ping")
logger.info("Titan v2 OK? shape=%s dtype=%s first3=%s", e.shape, e.dtype, e[:3])

INFO:RAG-Notebook:Titan v2 OK? shape=(1024,) dtype=float32 first3=[-0.01335682  0.01548294 -0.04272255]


In [ ]:
# Classe de indexação do Faiss
class FaissRAGIndex:
    def __init__(self, dim: int = EMBED_DIM):
        # Inner Product (IP) + normalização -> cosine
        self.index = faiss.IndexFlatIP(dim)
        self.texts: List[str] = []
        self.meta: List[Dict[str, Any]] = []
        self.vecs: np.ndarray | None = None

    def add(self, vectors: np.ndarray, texts: List[str], metas: List[Dict[str, Any]]):
        assert vectors.shape[0] == len(texts) == len(metas), "Mis-match vetores/textos/metas"
        # normaliza para cosine
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        vectors = vectors / np.clip(norms, 1e-12, None)

        if self.vecs is None:
            self.vecs = vectors
        else:
            self.vecs = np.vstack([self.vecs, vectors])

        self.index.add(vectors)
        self.texts.extend(texts)
        self.meta.extend(metas)

    def search(self, query: str, k: int = 5):
        q = titan_embed(query)
        q = q / np.clip(np.linalg.norm(q), 1e-12, None)
        distances, indices = self.index.search(q.reshape(1, -1), k)
        hits = []
        for score, idx in zip(distances[0].tolist(), indices[0].tolist()):
            if idx == -1:
                continue
            hits.append({"score": score, "text": self.texts[idx], "meta": self.meta[idx]})
        return hits


rag_index = FaissRAGIndex()


def load_s3_json_or_jsonl(bucket: str, key: str) -> List[Dict[str, Any]]:
    """
    Retorna lista de dicts (suporta JSON e JSONL).
    """
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw = obj["Body"].read().decode("utf-8")
    try:
        data = json.loads(raw)
        if isinstance(data, dict):
            data = [data]
        assert isinstance(data, list)
        return data
    except json.JSONDecodeError:
        # JSONL
        items = []
        for line in raw.splitlines():
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
        return items


def dict_to_text(d: Dict[str, Any]) -> str:
    """
    Flatten de JSON: gera "chave.subchave: valor" por linha.
    """
    lines: List[str] = []

    def _flat(prefix: str, obj: Any):
        if isinstance(obj, dict):
            for k, v in obj.items():
                _flat(f"{prefix}{k}.", v)
        elif isinstance(obj, list):
            for i, v in enumerate(obj):
                _flat(f"{prefix}{i}.", v)
        else:
            lines.append(f"{prefix[:-1]}: {obj}")

    _flat("", d)
    return "\n".join(lines)


def split_chunks(text: str, max_chars: int = 1500) -> List[str]:
    return [text[i : i + max_chars] for i in range(0, len(text), max_chars)]

In [12]:
BUCKET = "talk-to-your-data-bucket"
KEY = "queries_rag_sql.json"  # ou .jsonl


def build_rag_index_from_s3(bucket: str, key: str, max_chars: int = 1200, force: bool = False):
    """
    Constrói o índice FAISS a partir do S3. Se force=False e índice já tem vetores, pula.
    """
    if rag_index.index.ntotal > 0 and not force:
        logger.info("Índice já populado (ntotal=%d). Pulando rebuild.", rag_index.index.ntotal)
        return rag_index.index.ntotal

    logger.info("Carregando do S3 s3://%s/%s ...", bucket, key)
    data = load_s3_json_or_jsonl(bucket, key)
    logger.info("Objetos carregados: %d", len(data))

    n_chunks_total = 0
    t0 = time.time()

    for i, item in enumerate(data):
        txt = dict_to_text(item)
        chunks = split_chunks(txt, max_chars=max_chars)
        if not chunks:
            continue
        vecs = np.vstack([titan_embed(c) for c in chunks])
        metas = [{"doc_id": i, "chunk_id": j} for j in range(len(chunks))]
        rag_index.add(vecs, chunks, metas)
        n_chunks_total += len(chunks)

        if (i + 1) % 10 == 0:
            logger.info(
                "Indexados %d/%d docs ... (chunks até agora: %d)", i + 1, len(data), n_chunks_total
            )

    logger.info(
        "RAG pronto: ntotal=%d, texts=%d, tempo=%.1fs",
        rag_index.index.ntotal,
        len(rag_index.texts),
        time.time() - t0,
    )

    # Sanidade
    assert (
        rag_index.index.ntotal == len(rag_index.texts) == n_chunks_total
    ), "Inconsistência vetores/textos."

    return rag_index.index.ntotal


# --- Execute o bootstrap ---
_ = build_rag_index_from_s3(BUCKET, KEY, max_chars=1200, force=False)

print("ntotal:", rag_index.index.ntotal, "texts:", len(rag_index.texts))

INFO:RAG-Notebook:Carregando do S3 s3://talk-to-your-data-bucket/queries_rag_sql.json ...
INFO:RAG-Notebook:Objetos carregados: 10
INFO:RAG-Notebook:Indexados 10/10 docs ... (chunks até agora: 10)
INFO:RAG-Notebook:RAG pronto: ntotal=10, texts=10, tempo=2.8s


ntotal: 10 texts: 10


In [ ]:
# Tool rag_context
try:
    from strands import tool
except Exception:

    def tool(func=None, **_kwargs):
        def wrap(f):
            return f

        return wrap if func is None else wrap(func)


@tool
def rag_context(question: str, top_k: int = 5, max_chars_per_chunk: int = 800) -> dict:
    """
    Recupera trechos relevantes (contexto) para orientar a geração de SQL.
    Trunca cada chunk para evitar estourar tokens do LLM.
    """
    if rag_index.index.ntotal == 0:
        return {"error": "RAG index vazio. Rode o bootstrap primeiro."}

    hits = rag_index.search(question, k=top_k)
    results = []
    for h in hits:
        txt = h["text"]
        if len(txt) > max_chars_per_chunk:
            txt = txt[:max_chars_per_chunk] + "…"
        results.append({"text": txt, "score": float(h["score"]), "meta": h["meta"]})
    return {"question": question, "top_k": top_k, "context": results}

In [14]:
test_q = "Quais produtos ativos no arquivo e como relaciono com vendas mensais?"
res = rag_context(question=test_q, top_k=5, max_chars_per_chunk=800)
print(json.dumps(res, ensure_ascii=False, indent=2))

# Sanidade: garantir que trouxe algo
if "context" in res:
    print("Context chunks retornados:", len(res["context"]))

{
  "question": "Quais produtos ativos no arquivo e como relaciono com vendas mensais?",
  "top_k": 5,
  "context": [
    {
      "text": "id: Q04\nname: Evolução mensal de acessos\ndescription: Consolida acessos por mês para análise de sazonalidade.\nsql: \nSELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes\nFROM acessos_web\nGROUP BY mes\nORDER BY mes ASC;\n",
      "score": 0.32050687074661255,
      "meta": {
        "doc_id": 3,
        "chunk_id": 0
      }
    },
    {
      "text": "id: Q10\nname: Crescimento m/m (month-over-month)\ndescription: Calcula o crescimento percentual de acessos mês contra mês.\nsql: \nWITH mensal AS (\n  SELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes\n  FROM acessos_web\n  GROUP BY mes\n),\ncalc AS (\n  SELECT mes,\n         acessos_mes,\n         LAG(acessos_mes) OVER (ORDER BY mes) AS acessos_mes_prev\n  FROM mensal\n)\nSELECT mes,\n       acessos_mes,\n       acessos_mes_prev,\n       CAS

In [15]:
rag_context("Qual o crescimento em porcentagem de acessos mês a mês")

{'question': 'Qual o crescimento em porcentagem de acessos mês a mês',
 'top_k': 5,
 'context': [{'text': "id: Q10\nname: Crescimento m/m (month-over-month)\ndescription: Calcula o crescimento percentual de acessos mês contra mês.\nsql: \nWITH mensal AS (\n  SELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes\n  FROM acessos_web\n  GROUP BY mes\n),\ncalc AS (\n  SELECT mes,\n         acessos_mes,\n         LAG(acessos_mes) OVER (ORDER BY mes) AS acessos_mes_prev\n  FROM mensal\n)\nSELECT mes,\n       acessos_mes,\n       acessos_mes_prev,\n       CASE WHEN acessos_mes_prev IS NULL OR acessos_mes_prev = 0 THEN NULL\n            ELSE ROUND(100.0 * (acessos_mes - acessos_mes_prev) / acessos_mes_prev, 2)\n       END AS crescimento_percentual\nFROM calc\nORDER BY mes;\n",
   'score': 0.7190948724746704,
   'meta': {'doc_id': 9, 'chunk_id': 0}},
  {'text': "id: Q04\nname: Evolução mensal de acessos\ndescription: Consolida acessos por mês para análise de sazonalidad

## Text to SQL

In [16]:
import os
import logging
import boto3
from strands import tool

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# Helper para Redshift Data API
class RedshiftDataAPI:
    def __init__(self):
        self.region = os.getenv("AWS_REGION", "us-east-1")
        self.workgroupname = os.getenv(
            "REDSHIFT_WORKGROUP_NAME"
        )  # para cluster provisionado OU workgroup (serverless)
        self.database = os.getenv("REDSHIFT_DATABASE", "dev")
        self.db_user = os.getenv("REDSHIFT_USER")  # opcional para Data API
        self.use_serverless = os.getenv("USE_SERVERLESS", "false").lower() == "true"
        self.database = os.getenv("REDSHIFT_DATABASE")
        self.client = boto3.client("redshift-data", region_name=self.region)

    def _params(self, sql: str):
        p = {
            "Database": self.database,
            "Sql": sql,
        }
        if self.use_serverless:
            # Serverless usa WorkgroupName
            p["WorkgroupName"] = self.workgroupname
        elif self.db_user:
            p["DbUser"] = self.db_user
        return p

    def execute(self, sql: str, timeout: int = 60, poll_interval: float = 0.5) -> dict:
        """Executa via Data API e retorna dict {columns, rows}"""
        logger.info("Executando Redshift Data API...")
        resp = self.client.execute_statement(**self._params(sql))
        statement_id = resp["Id"]

        start = time.time()
        while True:
            desc = self.client.describe_statement(Id=statement_id)
            status = desc["Status"]
            if status in ("FINISHED", "FAILED", "ABORTED"):
                break
            if time.time() - start > timeout:
                raise TimeoutError("Timeout aguardando execução da Data API")
            time.sleep(poll_interval)

        if status != "FINISHED":
            raise RuntimeError(f"Data API terminou com status {status}: {desc.get('Error')}")

        res = self.client.get_statement_result(Id=statement_id)
        # Converter colunas e linhas
        cols = [m["name"] for m in res.get("ColumnMetadata", [])]
        rows = []
        for record in res.get("Records", []):
            row = []
            for col in record:
                # Cada col é um dict com uma única chave: stringValue/longValue/etc.
                val = next(iter(col.values()))
                row.append(val)
            rows.append(row)
        return {"columns": cols, "rows": rows}


# Instância compartilhada
_redshift = RedshiftDataAPI()

INFO:botocore.credentials:Found credentials in environment variables.


In [17]:
# Validação segura (sem DDL/DML) e LIMIT automático
FORBIDDEN = [
    r"\binsert\b",
    r"\bupdate\b",
    r"\bdelete\b",
    r"\bdrop\b",
    r"\balter\b",
    r"\bcreate\b",
    r"\btruncate\b",
    r"\bgrant\b",
    r"\brevoke\b",
    r"\bcopy\b",
    r"\bunload\b",
]

In [18]:
def validate_select(sql: str) -> str:
    s = sql.strip().lower()
    if not s.startswith("select"):
        raise ValueError("A consulta deve ser apenas SELECT.")
    if ";" in s[:-1]:
        raise ValueError("Evite múltiplas instruções em um único comando.")
    for kw in FORBIDDEN:
        if re.search(kw, s):
            raise ValueError("Palavra-chave não permitida na consulta.")
    if "limit" not in s:
        sql = sql.rstrip() + " LIMIT 100"
    return sql

In [ ]:
# Tool: Schema do Redshift
@tool
def redshift_schema() -> str:
    """
    Retorna o esquema do banco Redshift (schema.tabela e colunas).
    Exclui schemas do sistema.
    """
    sql = """
    SELECT table_schema, table_name, column_name, data_type, ordinal_position
    FROM information_schema.columns
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name, ordinal_position
    """
    result = _redshift.execute(sql)
    # Agrupar por schema.tabela
    from collections import defaultdict

    tables = defaultdict(list)
    for row in result["rows"]:
        schema, tname, col, dtype, _ord = row
        tables[(schema, tname)].append((col, dtype))

    lines = []
    for (schema, tname), cols in tables.items():
        col_desc = ", ".join(f"{c} {d}" for c, d in cols)
        lines.append(f"- {schema}.{tname}({col_desc})")
    return "Esquema do banco (Redshift):\n" + "\n".join(lines)

In [ ]:
# Ferramenta: Executar SELECT no Redshift


@tool
def run_redshift_select(sql: str) -> dict:
    """
    Executa uma consulta SELECT segura no Redshift via Data API e retorna colunas + linhas.
    """
    try:
        # Validação e ajuste (adiciona LIMIT se não houver)
        sql2 = validate_select(sql)
        result = _redshift.execute(sql2)
        # Padronizar a saída como no SQLite
        return {"sql": sql2, "columns": result["columns"], "rows": result["rows"]}
    except Exception as e:
        # Em caso de erro, retorna mensagem e a SQL original para debug
        return {"error": str(e), "sql": sql}

In [21]:
test = _redshift.execute("SELECT 1 AS ok")
print(test)  # esperado: {'columns': ['ok'], 'rows': [[1]]}

INFO:__main__:Executando Redshift Data API...


{'columns': ['ok'], 'rows': [[1]]}


In [22]:
# -----------------------------------------------------------------------------
# Agente Text-to-SQL e orquestrador
# -----------------------------------------------------------------------------
agent_text_to_sql = Agent(
    model=bedrock_model,
    system_prompt=TEXT_TO_SQL_PROMPT_V4,
    tools=[redshift_schema, run_redshift_select, rag_context],
)

In [23]:
agent_rag = Agent(model=bedrock_model, system_prompt=RAG_PROMPT_V1, tools=[])

agent_fallback = Agent(model=bedrock_model, system_prompt=FALLBACK_PROMPT_V1)

In [24]:
@tool
def call_text_to_sql(user_query: str) -> Dict[str, Any]:
    """
    Chama o agente Text-to-SQL com a pergunta do usuário.
    """
    return agent_text_to_sql(user_query)


@tool
def call_agent_rag(user_query: str) -> Dict[str, Any]:
    """
    Chama o agente RAG com a pergunta do usuário.
    """
    return agent_rag(user_query)


@tool
def call_agent_fallback(user_query: str) -> Dict[str, Any]:
    """
    Chama o agente fallback com a pergunta do usuário.
    """
    return agent_fallback(user_query)

In [25]:
orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=AGENT_ORCHESTRATOR_SYSTEM_PROMPT_V4,
    tools=[call_text_to_sql, call_agent_rag, call_agent_fallback],
)

In [26]:
agent_text_to_sql("Qual a quantidade de acessos total do mês de setembro de 2025")

INFO:strands.telemetry.metrics:Creating Strands MetricsClient


Vou ajudá-lo a encontrar a quantidade de acessos total do mês de setembro de 2025. Primeiro, vou buscar contexto relevante e depois consultar o esquema do banco.
Tool #1: rag_context


INFO:__main__:Executando Redshift Data API...



Tool #2: redshift_schema


INFO:__main__:Executando Redshift Data API...


Agora vou gerar a consulta SQL para obter a quantidade total de acessos do mês de setembro de 2025. Com base no contexto e esquema, vou usar a tabela `public.base_mockada_acessos` que contém as colunas `date` e `qtd_acessos`.
Tool #3: run_redshift_select


INFO:__main__:Executando Redshift Data API...


Parece que houve um erro. Vou tentar uma abordagem diferente usando a função DATE_TRUNC para filtrar o mês de setembro de 2025:
Tool #4: run_redshift_select
## Resposta

**Pergunta:** Qual a quantidade de acessos total do mês de setembro de 2025

**SQL utilizada:**
```sql
SELECT SUM(qtd_acessos) AS total_acessos_setembro_2025
FROM public.base_mockada_acessos
WHERE DATE_TRUNC('month', date) = '2025-09-01'
```

**Resultado:** A quantidade total de acessos no mês de setembro de 2025 foi de **888.874 acessos**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "## Resposta\n\n**Pergunta:** Qual a quantidade de acessos total do mês de setembro de 2025\n\n**SQL utilizada:**\n```sql\nSELECT SUM(qtd_acessos) AS total_acessos_setembro_2025\nFROM public.base_mockada_acessos\nWHERE DATE_TRUNC('month', date) = '2025-09-01'\n```\n\n**Resultado:** A quantidade total de acessos no mês de setembro de 2025 foi de **888.874 acessos**."}]}, metrics=EventLoopMetrics(cycle_count=5, tool_metrics={'rag_context': ToolMetrics(tool={'toolUseId': 'tooluse_VN89HydwScWYB1ApGC3DfA', 'name': 'rag_context', 'input': {'question': 'quantidade de acessos total mês setembro 2025'}}, call_count=1, success_count=1, error_count=0, total_time=0.542626142501831), 'redshift_schema': ToolMetrics(tool={'toolUseId': 'tooluse_Cig9NkCXTl-qV3UO163URQ', 'name': 'redshift_schema', 'input': {}}, call_count=1, success_count=1, error_count=0, total_time=3.6491127014160156), 'run_redshift_select': ToolMet

In [27]:
agent_text_to_sql("Qual o crescimento em porcentagem de acessos mês a mês.")

Vou buscar contexto relevante sobre crescimento mensal de acessos e depois gerar a consulta SQL apropriada.
Tool #5: rag_context
Perfeito! O contexto me forneceu exatamente a consulta para calcular o crescimento mês a mês. Vou adaptar a consulta para usar a tabela correta do esquema (`public.base_mockada_acessos`):
Tool #6: run_redshift_select


INFO:__main__:Executando Redshift Data API...


Vou tentar novamente com uma consulta mais simples, usando uma subquery em vez de CTE:
Tool #7: run_redshift_select
## Resposta

**Pergunta:** Qual o crescimento em porcentagem de acessos mês a mês

**SQL utilizada:**
```sql
SELECT mes,
       acessos_mes,
       acessos_mes_prev,
       CASE WHEN acessos_mes_prev IS NULL OR acessos_mes_prev = 0 THEN NULL
            ELSE ROUND(100.0 * (acessos_mes - acessos_mes_prev) / acessos_mes_prev, 2)
       END AS crescimento_percentual
FROM (
  SELECT mes,
         acessos_mes,
         LAG(acessos_mes) OVER (ORDER BY mes) AS acessos_mes_prev
  FROM (
    SELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes
    FROM public.base_mockada_acessos
    GROUP BY mes
  ) mensal
) calc
ORDER BY mes
```

**Resultados do crescimento mensal:**

- **Setembro 2025**: 888.874 acessos (primeiro mês, sem comparação)
- **Outubro 2025**: 1.268.436 acessos - **+42,70%** de crescimento
- **Novembro 2025**: 1.177.662 acessos - **-7,16%** (

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "## Resposta\n\n**Pergunta:** Qual o crescimento em porcentagem de acessos mês a mês\n\n**SQL utilizada:**\n```sql\nSELECT mes,\n       acessos_mes,\n       acessos_mes_prev,\n       CASE WHEN acessos_mes_prev IS NULL OR acessos_mes_prev = 0 THEN NULL\n            ELSE ROUND(100.0 * (acessos_mes - acessos_mes_prev) / acessos_mes_prev, 2)\n       END AS crescimento_percentual\nFROM (\n  SELECT mes,\n         acessos_mes,\n         LAG(acessos_mes) OVER (ORDER BY mes) AS acessos_mes_prev\n  FROM (\n    SELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes\n    FROM public.base_mockada_acessos\n    GROUP BY mes\n  ) mensal\n) calc\nORDER BY mes\n```\n\n**Resultados do crescimento mensal:**\n\n- **Setembro 2025**: 888.874 acessos (primeiro mês, sem comparação)\n- **Outubro 2025**: 1.268.436 acessos - **+42,70%** de crescimento\n- **Novembro 2025**: 1.177.662 acessos - **-7,16%** 

In [28]:
orchestrator_agent("Qual o crescimento em porcentagem de acessos mês a mês.")

Vou encaminhar sua pergunta sobre o crescimento percentual de acessos mês a mês para o agente Text-to-SQL, pois esta consulta envolve análise de dados estruturados com cálculos de métricas e agregações temporais.
Tool #1: call_text_to_sql
Vou buscar o crescimento percentual de acessos mês a mês novamente.
Tool #8: rag_context


INFO:__main__:Executando Redshift Data API...



Tool #9: run_redshift_select
## Resposta

**Pergunta:** Qual o crescimento em porcentagem de acessos mês a mês

**SQL utilizada:**
```sql
SELECT mes,
       acessos_mes,
       acessos_mes_prev,
       CASE WHEN acessos_mes_prev IS NULL OR acessos_mes_prev = 0 THEN NULL
            ELSE ROUND(100.0 * (acessos_mes - acessos_mes_prev) / acessos_mes_prev, 2)
       END AS crescimento_percentual
FROM (
  SELECT mes,
         acessos_mes,
         LAG(acessos_mes) OVER (ORDER BY mes) AS acessos_mes_prev
  FROM (
    SELECT DATE_TRUNC('month', date)::date AS mes, SUM(qtd_acessos) AS acessos_mes
    FROM public.base_mockada_acessos
    GROUP BY mes
  ) mensal
) calc
ORDER BY mes
```

**Crescimento mensal de acessos:**

| Mês | Acessos | Mês Anterior | Crescimento % |
|-----|---------|--------------|---------------|
| **Set/2025** | 888.874 | - | (primeiro mês) |
| **Out/2025** | 1.268.436 | 888.874 | **+42,70%** |
| **Nov/2025** | 1.177.662 | 1.268.436 | **-7,16%** |
| **Dez/2025** | 319.035

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Com base na análise dos dados, aqui está o crescimento percentual de acessos mês a mês:\n\n## Crescimento Mensal de Acessos\n\n| Mês | Total de Acessos | Crescimento % |\n|-----|------------------|---------------|\n| **Setembro/2025** | 888.874 | (primeiro mês) |\n| **Outubro/2025** | 1.268.436 | **+42,70%** |\n| **Novembro/2025** | 1.177.662 | **-7,16%** |\n| **Dezembro/2025** | 319.035 | **-72,91%** |\n\n### Principais Insights:\n\n1. **Outubro** apresentou o maior crescimento com **+42,70%** em relação a setembro\n2. **Novembro** teve uma pequena queda de **-7,16%** comparado a outubro\n3. **Dezembro** registrou uma queda significativa de **-72,91%** em relação a novembro\n\nA tendência mostra um pico em outubro, seguido de declínios nos meses subsequentes, sendo dezembro o mês com menor volume de acessos no período analisado.'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'call_text_